In [1]:
import sqlite3
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import ttest_ind, chi2_contingency
import numpy as np

# Connect to SQLite database
conn = sqlite3.connect('mydatabase.db')

In [2]:
# Load data from SQLite database into DataFrame.
# 'calls' is the view from part-3-SQL: the main table with the previous-outcome table
# joined on the call id. ORDER BY keeps the rows in the same order on every run.
query = """
SELECT *
FROM calls
ORDER BY id
"""
df = pd.read_sql_query(query, conn)
conn.close()

# The id identifies a call. It says nothing about the person, so it must not reach the models.
df = df.drop(columns=['id'])

print("Rows:", len(df))
print(f"Said yes: {df['y'].sum()} ({df['y'].mean():.1%})")

Rows: 43193
Said yes: 5021 (11.6%)


In [3]:
# Statistical Analysis: Hypothesis Testing
# T-test on 'balance' by 'poutcome'.
# Welch's version, because the two groups differ in size and in spread.
success = df.loc[df['poutcome'] == 'success', 'balance']
failure = df.loc[df['poutcome'] == 'failure', 'balance']
t_stat, p_val = ttest_ind(success, failure, equal_var=False)
print(f"T-test between success and failure balance: T-stat={t_stat:.4f}, P-value={p_val:.3g}")
print(f"Average balance: success {success.mean():.0f} ({len(success)} people), failure {failure.mean():.0f} ({len(failure)} people)")

T-test between success and failure balance: T-stat=4.2858, P-value=1.91e-05
Average balance: success 1921 (1424 people), failure 1462 (4709 people)


In [4]:
# Chi-squared tests: does saying yes go with each of these, or could the pattern be chance?
for column in ['poutcome', 'housing', 'job', 'education']:
    table = pd.crosstab(df[column], df['y'])
    chi2, p, dof, expected = chi2_contingency(table)
    print(f"{column}: chi2={chi2:.1f}, P-value={p:.3g}, degrees of freedom={dof}")

poutcome: chi2=4027.7, P-value=0, degrees of freedom=2
housing: chi2=825.3, P-value=1.72e-181, degrees of freedom=1
job: chi2=772.5, P-value=1.69e-159, degrees of freedom=10
education: chi2=233.4, P-value=2.08e-51, degrees of freedom=2


In [5]:
# The same thing in plain numbers: the share who said yes in each group
for column in ['poutcome', 'housing']:
    print(df.groupby(column)['y'].agg(share_yes='mean', people='count').round(3))
    print()

          share_yes  people
poutcome                   
failure       0.125    4709
other         0.095   37060
success       0.644    1424

         share_yes  people
housing                   
no           0.166   18901
yes          0.077   24292



In [6]:
# Machine Learning Integration: Classification
# Preprocessing
X = pd.get_dummies(df.drop('y', axis=1), drop_first=True)
y = df['y']

# stratify=y keeps the same share of yes in the training rows and the test rows
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print(f"Share of yes: training {y_train.mean():.4f}, test {y_test.mean():.4f}")

Training rows: 34554 | Test rows: 8639
Share of yes: training 0.1163, test 0.1162


In [7]:
# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# Logistic Regression
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)
predictions_lr = model_lr.predict(X_test_scaled)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, predictions_lr))

Logistic Regression Classification Report:


              precision    recall  f1-score   support

           0       0.92      0.97      0.95      7635
           1       0.65      0.36      0.46      1004

    accuracy                           0.90      8639
   macro avg       0.78      0.67      0.70      8639
weighted avg       0.89      0.90      0.89      8639



In [9]:
# Decision Tree
model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)
predictions_dt = model_dt.predict(X_test)
print("Decision Tree Classification Report:")
print(classification_report(y_test, predictions_dt))

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.92      0.93      7635
           1       0.45      0.49      0.47      1004

    accuracy                           0.87      8639
   macro avg       0.69      0.70      0.70      8639
weighted avg       0.88      0.87      0.87      8639



In [10]:
# Gradient Boosting
model_gb = GradientBoostingClassifier(random_state=42)
model_gb.fit(X_train_scaled, y_train)
predictions_gb = model_gb.predict(X_test_scaled)
print("Gradient Boosting Classification Report:")
print(classification_report(y_test, predictions_gb))

Gradient Boosting Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      7635
           1       0.65      0.42      0.51      1004

    accuracy                           0.91      8639
   macro avg       0.79      0.69      0.73      8639
weighted avg       0.89      0.91      0.90      8639



In [11]:
# Model Evaluation
accuracy_lr = accuracy_score(y_test, predictions_lr)
accuracy_dt = accuracy_score(y_test, predictions_dt)
accuracy_gb = accuracy_score(y_test, predictions_gb)
print(f"Accuracies:\nLogistic Regression: {accuracy_lr}\nDecision Tree: {accuracy_dt}\nGradient Boosting: {accuracy_gb}")

# What accuracy is worth here: a model that says no to everybody
accuracy_always_no = 1 - y_test.mean()
print(f"Always predicting no: {accuracy_always_no}")

Accuracies:
Logistic Regression: 0.9026507697650191
Decision Tree: 0.8712813983099896
Gradient Boosting: 0.9063549021877532
Always predicting no: 0.8837828452367172


In [12]:
# Accuracy cannot tell these models apart, so score them on the yes class as well.
# ROC AUC is the chance that the model ranks someone who said yes above someone who said no.
rows = []
for name, model, test_features, predictions in [
    ('Logistic Regression', model_lr, X_test_scaled, predictions_lr),
    ('Decision Tree', model_dt, X_test, predictions_dt),
    ('Gradient Boosting', model_gb, X_test_scaled, predictions_gb),
]:
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, predictions, average='binary')
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, predictions),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(y_test, model.predict_proba(test_features)[:, 1]),
    })

results = pd.DataFrame(rows).set_index('model')
print(results.round(4))

                     accuracy  precision  recall      f1  roc_auc
model                                                            
Logistic Regression    0.9027     0.6468  0.3576  0.4606   0.9099
Decision Tree          0.8713     0.4501  0.4851  0.4669   0.7036
Gradient Boosting      0.9064     0.6507  0.4193  0.5100   0.9164


In [13]:
# What the best model leans on
importance = pd.Series(model_gb.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(10).round(4))

duration            0.5035
poutcome_success    0.1968
pdays               0.0664
housing_yes         0.0494
age                 0.0476
month_mar           0.0452
month_oct           0.0162
month_sep           0.0134
balance             0.0099
month_may           0.0075
dtype: float64


In [14]:
# 'duration' is how long the call lasted, and that is only known once the call is over.
# A model that is meant to choose who to call cannot use it.
# Same rows, same model, without that one column:
X_train_nd = X_train.drop(columns=['duration'])
X_test_nd = X_test.drop(columns=['duration'])

scaler_nd = StandardScaler()
model_gb_nd = GradientBoostingClassifier(random_state=42)
model_gb_nd.fit(scaler_nd.fit_transform(X_train_nd), y_train)

test_nd_scaled = scaler_nd.transform(X_test_nd)
predictions_nd = model_gb_nd.predict(test_nd_scaled)
precision_nd, recall_nd, f1_nd, _ = precision_recall_fscore_support(y_test, predictions_nd, average='binary')

comparison = pd.DataFrame({
    'with duration': results.loc['Gradient Boosting', ['roc_auc', 'f1', 'recall', 'precision', 'accuracy']],
    'without duration': pd.Series({
        'roc_auc': roc_auc_score(y_test, model_gb_nd.predict_proba(test_nd_scaled)[:, 1]),
        'f1': f1_nd,
        'recall': recall_nd,
        'precision': precision_nd,
        'accuracy': accuracy_score(y_test, predictions_nd),
    }),
})
comparison['change'] = comparison['without duration'] - comparison['with duration']
print(comparison.round(4))

           with duration  without duration  change
roc_auc           0.9164            0.7645 -0.1518
f1                0.5100            0.3133 -0.1967
recall            0.4193            0.2082 -0.2112
precision         0.6507            0.6333 -0.0174
accuracy          0.9064            0.8940 -0.0124


In [15]:
# Insights: Summarize Key Findings
best = results['roc_auc'].idxmax()
yes_by_outcome = df.groupby('poutcome')['y'].mean()
yes_by_housing = df.groupby('housing')['y'].mean()

print("Key Insights:")
print(f"- {df['y'].mean():.1%} of the calls ended in a yes, so always predicting no is {accuracy_always_no:.1%} accurate. Accuracy cannot rank these models.")
print(f"- By ROC AUC the best model is {best} at {results.loc[best, 'roc_auc']:.3f}. Logistic Regression is at {results.loc['Logistic Regression', 'roc_auc']:.3f} and the unpruned Decision Tree at {results.loc['Decision Tree', 'roc_auc']:.3f}.")
print(f"- {importance.index[0]} carries {importance.iloc[0]:.0%} of the Gradient Boosting model's decisions, and it is only known after the call. Without it ROC AUC falls to {comparison.loc['roc_auc', 'without duration']:.3f}.")
print(f"- People whose previous campaign succeeded said yes {yes_by_outcome['success']:.0%} of the time. People with a housing loan said yes {yes_by_housing['yes']:.1%} of the time, against {yes_by_housing['no']:.1%} without one.")

Key Insights:
- 11.6% of the calls ended in a yes, so always predicting no is 88.4% accurate. Accuracy cannot rank these models.
- By ROC AUC the best model is Gradient Boosting at 0.916. Logistic Regression is at 0.910 and the unpruned Decision Tree at 0.704.
- duration carries 50% of the Gradient Boosting model's decisions, and it is only known after the call. Without it ROC AUC falls to 0.765.
- People whose previous campaign succeeded said yes 64% of the time. People with a housing loan said yes 7.7% of the time, against 16.6% without one.
